In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
# Load Data
df1 = pd.read_csv('credit_record.csv')
df2 = pd.read_csv('application_record.csv')

In [3]:
df1.shape

(1048575, 3)

In [4]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 3 columns):
 #   Column          Non-Null Count    Dtype 
---  ------          --------------    ----- 
 0   ID              1048575 non-null  int64 
 1   MONTHS_BALANCE  1048575 non-null  int64 
 2   STATUS          1048575 non-null  object
dtypes: int64(2), object(1)
memory usage: 24.0+ MB


In [5]:
df1['STATUS'].unique()

array(['X', '0', 'C', '1', '2', '3', '4', '5'], dtype=object)

In [6]:
df1['MONTHS_BALANCE'].unique()

array([  0,  -1,  -2,  -3,  -4,  -5,  -6,  -7,  -8,  -9, -10, -11, -12,
       -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25,
       -26, -27, -28, -29, -30, -31, -32, -33, -34, -35, -36, -37, -38,
       -39, -40, -41, -42, -43, -44, -45, -46, -47, -48, -49, -50, -51,
       -52, -53, -54, -55, -56, -57, -58, -59, -60], dtype=int64)

In [7]:
#sort
df1 = df1.sort_values(['ID', 'MONTHS_BALANCE'])

In [8]:
#spit past & future data 
df_past = df1[df1['MONTHS_BALANCE'] >= -6]
df_future = df1[df1['MONTHS_BALANCE'] < -6]


In [9]:
# Feature Engineering
past_features = df_past.groupby('ID').agg({
    'STATUS': lambda x: (x.isin(['1','2','3','4','5'])).sum(),
    'MONTHS_BALANCE': 'count'
}).reset_index()


In [10]:
past_features.columns = ['ID', 'PAST_DELAY_COUNT', 'PAST_MONTH_COUNT']


In [11]:
past_features

,ID,PAST_DELAY_COUNT,PAST_MONTH_COUNT
0,5001711,0,4
1,5001712,0,7
2,5001713,0,7
3,5001714,0,7
4,5001715,0,7
...,...,...,...
36886,5150481,0,7
36887,5150483,0,7
36888,5150484,0,7
36889,5150485,0,2


In [12]:
past_features['PAST_DELAY_COUNT'].unique()

array([0, 4, 1, 2, 6, 3, 5, 7], dtype=int64)

In [13]:
#create target variable
df_future['label'] = df_future['STATUS'].isin(['2','3','4','5']).astype(int)
customer_label = df_future.groupby('ID')['label'].max().reset_index()

C:\Users\user\AppData\Local\Temp\ipykernel_6048\2972326578.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_future['label'] = df_future['STATUS'].isin(['2','3','4','5']).astype(int)


In [14]:
customer_label

,ID,label
0,5001712,0
1,5001713,0
2,5001714,0
3,5001715,0
4,5001717,0
...,...,...
40360,5150481,0
40361,5150482,0
40362,5150483,0
40363,5150484,0


In [15]:
# Merge dataset
df = pd.merge(df2, past_features, on='ID')
df = pd.merge(df, customer_label, on='ID')

In [16]:
df

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,...,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,PAST_DELAY_COUNT,PAST_MONTH_COUNT,label
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,...,-4542,1,1,0,0,NaN,2.0,0,7,0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,...,-4542,1,1,0,0,NaN,2.0,0,7,0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,...,-1134,1,0,0,0,Security staff,2.0,0,7,0
3,5008810,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,...,-3051,1,0,1,1,Sales staff,1.0,0,7,0
4,5008811,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,...,-3051,1,0,1,1,Sales staff,1.0,0,7,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22644,5149828,M,Y,Y,0,315000.0,Working,Secondary / secondary special,Married,House / apartment,...,-2420,1,0,0,0,Managers,2.0,4,7,0
22645,5149834,F,N,Y,0,157500.0,Commercial associate,Higher education,Married,House / apartment,...,-1325,1,0,1,1,Medicine staff,2.0,2,7,1
22646,5149838,F,N,Y,0,157500.0,Pensioner,Higher education,Married,House / apartment,...,-1325,1,0,1,1,Medicine staff,2.0,0,7,1
22647,5150049,F,N,Y,0,283500.0,Working,Secondary / secondary special,Married,House / apartment,...,-655,1,0,0,0,Sales staff,2.0,2,7,0


In [17]:
df.columns

Index(['ID', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN',
       'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
       'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'DAYS_BIRTH',
       'DAYS_EMPLOYED', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE',
       'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'PAST_DELAY_COUNT',
       'PAST_MONTH_COUNT', 'label'],
      dtype='object')

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22649 entries, 0 to 22648
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   22649 non-null  int64  
 1   CODE_GENDER          22649 non-null  object 
 2   FLAG_OWN_CAR         22649 non-null  object 
 3   FLAG_OWN_REALTY      22649 non-null  object 
 4   CNT_CHILDREN         22649 non-null  int64  
 5   AMT_INCOME_TOTAL     22649 non-null  float64
 6   NAME_INCOME_TYPE     22649 non-null  object 
 7   NAME_EDUCATION_TYPE  22649 non-null  object 
 8   NAME_FAMILY_STATUS   22649 non-null  object 
 9   NAME_HOUSING_TYPE    22649 non-null  object 
 10  DAYS_BIRTH           22649 non-null  int64  
 11  DAYS_EMPLOYED        22649 non-null  int64  
 12  FLAG_MOBIL           22649 non-null  int64  
 13  FLAG_WORK_PHONE      22649 non-null  int64  
 14  FLAG_PHONE           22649 non-null  int64  
 15  FLAG_EMAIL           22649 non-null 

In [19]:
# Data Cleaning
df['FLAG_MOBIL'].value_counts()

FLAG_MOBIL
1    22649
Name: count, dtype: int64

In [20]:
# this feature only have one category (1),means evryone have moblie phone 
# so we can remove it
df.drop('FLAG_MOBIL', axis=1, inplace=True)

In [21]:
df.isnull().sum()

ID                        0
CODE_GENDER               0
FLAG_OWN_CAR              0
FLAG_OWN_REALTY           0
CNT_CHILDREN              0
AMT_INCOME_TOTAL          0
NAME_INCOME_TYPE          0
NAME_EDUCATION_TYPE       0
NAME_FAMILY_STATUS        0
NAME_HOUSING_TYPE         0
DAYS_BIRTH                0
DAYS_EMPLOYED             0
FLAG_WORK_PHONE           0
FLAG_PHONE                0
FLAG_EMAIL                0
OCCUPATION_TYPE        6937
CNT_FAM_MEMBERS           0
PAST_DELAY_COUNT          0
PAST_MONTH_COUNT          0
label                     0
dtype: int64

In [22]:
df['OCCUPATION_TYPE'] = df['OCCUPATION_TYPE'].fillna('Unknown')

In [23]:
df['DAYS_EMPLOYED'].value_counts()

DAYS_EMPLOYED
 365243    3698
-1751        46
-401         46
-1539        40
-2531        39
           ... 
-2035         1
-3690         1
-10600        1
-4037         1
-11272        1
Name: count, Length: 3341, dtype: int64

In [24]:
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].fillna(df['DAYS_EMPLOYED'].median())

In [25]:
df['EMPLOYED_YEARS'] = abs(df['DAYS_EMPLOYED']) / 365
df.drop('DAYS_EMPLOYED', axis=1, inplace=True)

df['AGE'] = abs(df['DAYS_BIRTH']) / 365
df.drop('DAYS_BIRTH', axis=1, inplace=True)


In [26]:
X = df.drop(['label', 'ID'], axis=1)
y = df['label']

In [27]:
#Train Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [28]:
# pipline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_cols = [
    'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE'
]

numeric_cols = [col for col in X.columns if col not in categorical_cols]

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])


In [29]:
# SMOTE
# handle imblelence dataset
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [30]:
#model
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
}

In [31]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

best_model = None
best_score = 0

In [32]:
for name, model in models.items():
    print(f"Training {name}")

    pipeline = ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:,1]

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("ROC AUC:", roc_auc_score(y_test, y_prob))
    print(classification_report(y_test, y_pred))

    score = roc_auc_score(y_test, y_prob)

    if score > best_score:
        best_score = score
        best_model = pipeline


Training Logistic Regression
Accuracy: 0.6490066225165563
ROC AUC: 0.6038505625955735
              precision    recall  f1-score   support

           0       0.98      0.65      0.78      4437
           1       0.03      0.48      0.05        93

    accuracy                           0.65      4530
   macro avg       0.51      0.57      0.42      4530
weighted avg       0.96      0.65      0.77      4530

Training Random Forest
Accuracy: 0.9463576158940398
ROC AUC: 0.6836935738329444
              precision    recall  f1-score   support

           0       0.98      0.96      0.97      4437
           1       0.09      0.17      0.12        93

    accuracy                           0.95      4530
   macro avg       0.54      0.57      0.54      4530
weighted avg       0.96      0.95      0.95      4530



In [33]:
joblib.dump(best_model, 'pipeline.pkl')

print("Best model saved as pipeline.pkl")

Best model saved as pipeline.pkl


In [34]:
import os
print(os.getcwd())

c:\Users\user\Desktop\credit risk update
